# Image Colorization using Deep Exemplar-based Video Colorization
This notebook demonstrates how to colorize a single image using a reference image.

In [ ]:
import os
import cv2
import numpy as np
import torch
import torchvision.transforms as transform_lib
from PIL import Image
import matplotlib.pyplot as plt

import lib.TestTransforms as transforms
from models.ColorVidNet import ColorVidNet
from models.FrameColor import frame_colorization
from models.NonlocalNet import VGG19_pytorch, WarpNet
from utils.util import batch_lab2rgb_transpose_mc, tensor_lab2rgb, uncenter_l
from utils.util_distortion import CenterPad, Normalize, RGB2Lab, ToTensor

In [ ]:
# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load networks
nonlocal_net = WarpNet(1)
colornet = ColorVidNet(7)
vggnet = VGG19_pytorch()

# Load pre-trained weights
vggnet.load_state_dict(torch.load('data/vgg19_conv.pth', map_location=device))
for param in vggnet.parameters():
    param.requires_grad = False

nonlocal_net.load_state_dict(torch.load('checkpoints/video_moredata_l1/nonlocal_net_iter_76000.pth', map_location=device))
colornet.load_state_dict(torch.load('checkpoints/video_moredata_l1/colornet_iter_76000.pth', map_location=device))

nonlocal_net.eval()
colornet.eval()
vggnet.eval()

nonlocal_net.to(device)
colornet.to(device)
vggnet.to(device)
print("Models loaded successfully!")

In [ ]:
# Preprocessing transform
image_size = [216 * 2, 384 * 2]
transform = transforms.Compose([
    CenterPad(image_size),
    transform_lib.CenterCrop(image_size),
    RGB2Lab(),
    ToTensor(),
    Normalize()
])

def colorize_image(target_image_path, reference_image_path):
    # Load images
    target_img = Image.open(target_image_path).convert('RGB')
    ref_img = Image.open(reference_image_path).convert('RGB')
    
    # Process reference image
    IB_lab_large = transform(ref_img).unsqueeze(0).to(device)
    IB_lab = torch.nn.functional.interpolate(IB_lab_large, scale_factor=0.5, mode="bilinear", align_corners=False)
    
    with torch.no_grad():
        I_reference_lab = IB_lab
        I_reference_l = I_reference_lab[:, 0:1, :, :]
        I_reference_ab = I_reference_lab[:, 1:3, :, :]
        I_reference_rgb = tensor_lab2rgb(torch.cat((uncenter_l(I_reference_l), I_reference_ab), dim=1))
        features_B = vggnet(I_reference_rgb, ["r12", "r22", "r32", "r42", "r52"], preprocess=True)
        
    # Process target image
    IA_lab_large = transform(target_img).unsqueeze(0).to(device)
    IA_lab = torch.nn.functional.interpolate(IA_lab_large, scale_factor=0.5, mode="bilinear", align_corners=False)
    IA_l = IA_lab[:, 0:1, :, :]
    
    I_last_lab_predict = torch.zeros_like(IA_lab).to(device)

    # Colorization
    with torch.no_grad():
        I_current_ab_predict, _, _ = frame_colorization(
            IA_lab,
            I_reference_lab,
            I_last_lab_predict,
            features_B,
            vggnet,
            nonlocal_net,
            colornet,
            feature_noise=0,
            temperature=1e-10,
        )

    # Upsample and Post-process
    curr_bs_l = IA_lab_large[:, 0:1, :, :]
    curr_predict = torch.nn.functional.interpolate(I_current_ab_predict.data.cpu(), scale_factor=2, mode="bilinear", align_corners=False) * 1.25
    
    # Optional WLS filter can be applied here for smoothing (skipping for simplicity)
    IA_predict_rgb = batch_lab2rgb_transpose_mc(curr_bs_l[:32], curr_predict[:32, ...])
    return IA_predict_rgb[0]

print("Colorization function defined!")

In [ ]:
# Example usage:
target_path = 'sample_videos/clips/v32/1415.png'
reference_path = 'sample_videos/ref/v32/01.jpg'

if os.path.exists(target_path) and os.path.exists(reference_path):
    colored_image = colorize_image(target_path, reference_path)
    
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    
    img_target = Image.open(target_path).convert('L')
    ax[0].imshow(img_target, cmap='gray')
    ax[0].set_title('Target Grayscale')
    ax[0].axis('off')

    img_ref = Image.open(reference_path)
    ax[1].imshow(img_ref)
    ax[1].set_title('Reference Color')
    ax[1].axis('off')
    
    ax[2].imshow(colored_image)
    ax[2].set_title('Colorized Output')
    ax[2].axis('off')
    
    plt.show()
else:
    print("Sample images not found. Please provide valid paths to target and reference images.")